# Final Machine Learning · Agua en superficie después de la lluvia con radar NISAR

Imágenes NISAR (banda L, grilla de 5 m) tomadas como máximo 4 h después de una lluvia registrada en una estación de la
ciudad, en Perú, Ecuador y Chile. Para cada píxel se predice si hay agua con una U-Net y con modelos de machine learning; al
final, los charcos de cada lluvia y el mapa de Lima Sur.

0. Entorno · 1. Librerías · 2. Datos · 3. Preprocesamiento · 4. Normalización · 5. U-Net · 6. Validación ·
7. Machine learning · 8. Predicción y prueba externa · 9. Agua de la lluvia · 10. Charcos · 11. Mapas de Lima Sur ·
12. Archivos de salida

## 0. Entorno

In [ ]:
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

ENTRENAR = True
PRUEBA_RAPIDA = False
DATASET_KAGGLE = "andres4499/datafinalmlvol2"
EN_KAGGLE = Path("/kaggle/input").exists()


def buscar_data(raiz):
    hh = next(iter(sorted(raiz.glob("**/nisar/*/hh_*.tif"))), None)
    h5 = next(iter(sorted(raiz.glob("**/*.h5"))), None)
    if hh is None and h5 is None:
        raise FileNotFoundError(f"No encuentro imágenes NISAR (hh_*.tif ni *.h5) dentro de {raiz}")
    return (hh.parents[2] if hh else None), (h5.parents[1] if h5 else None)


if EN_KAGGLE:
    PAQUETES = {"segmentation_models_pytorch": "segmentation-models-pytorch==0.5.0", "rasterio": "rasterio", "h5py": "h5py", "xgboost": "xgboost", "folium": "folium"}
    faltan = [p for m, p in PAQUETES.items() if importlib.util.find_spec(m) is None]
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
    ENTRADA = next((p for p in sorted(Path("/kaggle/input").glob(f"**/{DATASET_KAGGLE.split('/')[1]}")) if p.is_dir()),
                   Path("/kaggle/input"))
    try:
        PROCESADO, CRUDOS = buscar_data(ENTRADA)
    except FileNotFoundError:
        import kagglehub
        PROCESADO, CRUDOS = buscar_data(Path(kagglehub.dataset_download(DATASET_KAGGLE)))
    TRABAJO = Path("/tmp/proyecto/Data")
    TRABAJO.mkdir(parents=True, exist_ok=True)
    if CRUDOS and not (TRABAJO / "crudos_nisar").exists():
        (TRABAJO / "crudos_nisar").symlink_to(CRUDOS)
    if PROCESADO and not (TRABAJO / "procesado").exists():
        shutil.copytree(PROCESADO, TRABAJO / "procesado")
    os.chdir(TRABAJO.parent)
    SALIDA = Path("/kaggle/working")
    print("dataset · procesado:", PROCESADO, "· crudos:", CRUDOS)
else:
    SALIDA = Path("Data/entrega")
Path("Data/procesado").mkdir(parents=True, exist_ok=True)
print("entorno:", "Kaggle" if EN_KAGGLE else "PC local", "| data:", Path("Data").resolve(), "| salida:", SALIDA.resolve())

## 1. Librerías

In [ ]:
import json
import re
import time
import unicodedata
import warnings
from functools import lru_cache
from pathlib import Path
import folium
import joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, h5py, rasterio, requests, pyproj, torch, segmentation_models_pytorch as smp
from matplotlib.colors import ListedColormap
from pyproj import Transformer
from rasterio.features import shapes
from rasterio.merge import merge
from rasterio.transform import array_bounds, from_origin, rowcol
from rasterio.warp import Resampling, calculate_default_transform, reproject, transform_geom
from rasterio.windows import Window
from scipy import ndimage
from scipy.interpolate import LinearNDInterpolator
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.base import clone
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score,
                             jaccard_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

pyproj.network.set_network_enabled(True)
warnings.filterwarnings("ignore")
SEMILLA = 0

## 2. Datos

Escenas por ciudad, pista y fecha, con un día seco por ciudad como referencia.

In [ ]:
SECOS = {"arequipa": "20261013", "cajamarca": "20260918", "chillan": "20260819", "concepcion": "20260819",
         "cuenca": "20251204", "cusco": "20260617", "guayaquil": "20260626", "iquitos": "20260624",
         "la_serena": "20251108", "lima": "20260825", "los_angeles": "20260819", "manta": "20260628",
         "osorno": "20261019", "pucallpa": "20260119", "puerto_montt": "20260819", "salinas": "20260616",
         "santiago": "20260701", "valdivia": "20251104"}
ROL = {**dict.fromkeys(["iquitos_2026-06-24", "iquitos_2026-09-04", "iquitos_2026-10-22", "concepcion_2026-09-24",
                        "puerto_montt_2026-07-15"], "val"),
       **dict.fromkeys(["lima_2026-11-05", "concepcion_2026-09-30", "valdivia_2026-07-08", "guayaquil_2026-11-12",
                        "osorno_2026-11-05"], "test"),
       **dict.fromkeys(["lima_2026-01-02", "lima_2026-07-06", "lima_2026-07-08", "lima_2026-07-20", "lima_2026-08-13",
                        "lima_2026-09-11", "lima_2026-10-17", "lima_2026-10-22", "lima_2026-10-24"], "garua"),
       **dict.fromkeys(["cuenca_2025-10-29", "cuenca_2025-11-22", "cuenca_2026-10-31", "cuenca_2026-11-17",
                        "cusco_2025-12-19", "cajamarca_2026-08-25", "cajamarca_2026-10-12", "arequipa_2026-01-22",
                        "arequipa_2026-02-15"], "prediccion")}
ZONA = {**dict.fromkeys(["lima", "arequipa", "iquitos", "pucallpa", "cusco", "cajamarca"], "America/Lima"),
        **dict.fromkeys(["guayaquil", "manta", "salinas", "cuenca"], "America/Guayaquil"),
        **dict.fromkeys(["santiago", "la_serena", "concepcion", "chillan", "los_angeles", "valdivia", "osorno", "puerto_montt"],
                        "America/Santiago")}

if any(Path("Data/crudos_nisar").glob("*/*.h5")):
    archivos = pd.DataFrame([{"h5": h, "ciudad": h.parent.name, "pista": h.name[21:26], "cuadro": h.name[27:30],
                              "fecha": h.name[43:51],
                              "hora_local": pd.to_datetime(h.name[43:58], format="%Y%m%dT%H%M%S").tz_localize("UTC")
                              .tz_convert(ZONA.get(h.parent.name, "America/Lima")).strftime("%H:%M")}
                             for h in sorted(Path("Data/crudos_nisar").glob("*/*.h5"))])
    escenas = archivos.groupby(["ciudad", "pista", "fecha"]).agg(h5=("h5", list), cuadros=("cuadro", "+".join),
                                                                  hora_local=("hora_local", "first")).reset_index()
    escenas.drop(columns="h5").to_csv("Data/procesado/escenas.csv", index=False)
else:
    if Path("Data/procesado/escenas.csv").exists():
        escenas = pd.read_csv("Data/procesado/escenas.csv", dtype=str)
    else:
        escenas = pd.DataFrame([dict(zip(["ciudad", "fecha", "pista"], p.stem[3:].rsplit("_", 2)), cuadros="", hora_local="")
                                for p in Path("Data/procesado/nisar").glob("*/hh_*.tif")])
        escenas["fecha"] = escenas.fecha.str.replace("-", "")
        escenas["pista"] = escenas.pista.str[:3] + "_" + escenas.pista.str[3:]
    escenas["h5"] = [[] for _ in range(len(escenas))]
escenas["base"] = escenas.ciudad + "_" + escenas.fecha.str[:4] + "-" + escenas.fecha.str[4:6] + "-" + escenas.fecha.str[6:]
escenas["clave"] = escenas.base + "_" + escenas.pista.str.replace("_", "")
escenas["dia"] = np.where(escenas.fecha == escenas.ciudad.map(SECOS), "seco", "lluvia")
escenas["rol"] = escenas.base.map(ROL).fillna("train")
escenas = escenas.sort_values(["ciudad", "dia", "fecha", "pista"]).reset_index(drop=True)
escenas[["clave", "pista", "cuadros", "hora_local", "dia", "rol"]]

In [ ]:
LLUVIAS, SECAS = escenas[escenas.dia == "lluvia"], escenas[escenas.dia == "seco"].set_index("ciudad")
POR_CIUDAD = (pd.crosstab(LLUVIAS.ciudad, LLUVIAS.rol).reindex(columns=["train", "val", "test", "garua", "prediccion"], fill_value=0)
              .assign(lluvias=LLUVIAS.groupby("ciudad").size(), fechas_lluvia=LLUVIAS.groupby("ciudad").fecha.agg(", ".join),
                      dia_seco=SECAS.fecha, rol_dia_seco=SECAS.rol))
display(escenas.groupby(["dia", "rol"]).size().unstack(fill_value=0))
POR_CIUDAD

## 3. Preprocesamiento

### 3.1 Ciudades

In [ ]:
CAJAS = {"lima": (-77.1694, -12.3083, -76.7589, -11.7924), "arequipa": (-71.6681, -16.4901, -71.4397, -16.2583),
         "iquitos": (-73.30, -3.80, -73.22, -3.70), "pucallpa": (-74.60, -8.42, -74.50, -8.34),
         "cusco": (-72.02, -13.56, -71.90, -13.50), "cajamarca": (-78.54, -7.19, -78.48, -7.13),
         "guayaquil": (-80.00, -2.30, -79.75, -2.05), "manta": (-80.85, -1.05, -80.55, -0.85), "salinas": (-80.95, -2.32, -80.70, -2.10),
         "cuenca": (-79.05, -2.93, -78.95, -2.86), "santiago": (-70.75, -33.52, -70.55, -33.38),
         "la_serena": (-71.40, -30.05, -71.15, -29.80), "concepcion": (-73.15, -36.95, -72.90, -36.70),
         "chillan": (-72.14, -36.64, -72.06, -36.58), "los_angeles": (-72.39, -37.50, -72.32, -37.44),
         "valdivia": (-73.35, -39.92, -73.15, -39.75), "osorno": (-73.17, -40.61, -73.09, -40.54),
         "puerto_montt": (-73.00, -41.49, -72.88, -41.44)}
EPSG = {"lima": 32718, "arequipa": 32719, "iquitos": 32718, "pucallpa": 32718, "cusco": 32719, "cajamarca": 32717,
        "guayaquil": 32717, "manta": 32717, "salinas": 32717, "cuenca": 32717, "santiago": 32719, "la_serena": 32719,
        "concepcion": 32718, "chillan": 32718, "los_angeles": 32718, "valdivia": 32718, "osorno": 32718, "puerto_montt": 32718}

### 3.2 Grilla de 5 m, DEM y WorldCover

In [ ]:
GEOIDE = Transformer.from_crs("EPSG:4326+3855", "EPSG:4979", always_xy=True)


def geoide(lon, lat):
    z = GEOIDE.transform(lon, lat, 0)[2]
    if not np.isfinite(z):
        from pyproj.datadir import get_user_data_dir
        grilla = Path(get_user_data_dir(True)) / "us_nga_egm08_25.tif"
        if not grilla.exists():
            pedido = requests.get("https://cdn.proj.org/us_nga_egm08_25.tif", timeout=900)
            pedido.raise_for_status()
            grilla.write_bytes(pedido.content)
        z = Transformer.from_crs("EPSG:4326+3855", "EPSG:4979", always_xy=True).transform(lon, lat, 0)[2]
    if not np.isfinite(z):
        raise RuntimeError("No pude calcular el geoide EGM2008: revisa que Internet esté en On")
    return float(z)


def pedir(metodo, url, **kw):
    for intento in range(8):
        try:
            r = requests.request(metodo, url, timeout=120, **kw)
            if r.status_code != 429 and r.status_code < 500:
                r.raise_for_status()
                return r.json()
            motivo = f"respondió {r.status_code}"
        except (requests.ConnectionError, requests.Timeout) as error:
            motivo = f"conexión: {error}"
        if intento == 7:
            raise RuntimeError(f"Planetary Computer no responde ({motivo}): {url}")
        print(f"Planetary Computer {motivo}; espero {30 * (intento + 1)} s y reintento")
        time.sleep(30 * (intento + 1))


@lru_cache
def token_pc(coleccion):
    return pedir("GET", f"https://planetarycomputer.microsoft.com/api/sas/v1/token/{coleccion}")["token"]


@lru_cache
def mosaico(coleccion, ciudad):
    o, s, e, n = CAJAS[ciudad]
    caja = [o - 0.02, s - 0.02, e + 0.02, n + 0.02]
    fecha = {"datetime": "2021-01-01T00:00:00Z/2021-12-31T23:59:59Z"} if coleccion == "esa-worldcover" else {}
    capa = "map" if coleccion == "esa-worldcover" else "data"
    items = pedir("POST", "https://planetarycomputer.microsoft.com/api/stac/v1/search",
                  json={"collections": [coleccion], "bbox": caja, **fecha})["features"]
    if not items:
        raise RuntimeError(f"Planetary Computer no devolvió {coleccion} para {ciudad}")
    for intento in range(3):
        try:
            arr, tr = merge([rasterio.open(i["assets"][capa]["href"] + "?" + token_pc(coleccion)) for i in items], bounds=caja)
            return arr[0], tr
        except Exception as error:
            if intento == 2:
                raise
            print(f"{coleccion} {ciudad}: falló la lectura ({error}); pido otro token y reintento")
            token_pc.cache_clear()
            time.sleep(30)


def en_grilla(coleccion, ciudad, metodo):
    arr, tr = mosaico(coleccion, ciudad)
    p = GRILLAS[ciudad]
    out = np.zeros((p["height"], p["width"]), arr.dtype)
    reproject(arr, out, src_transform=tr, src_crs="EPSG:4326", dst_transform=p["transform"], dst_crs=p["crs"], resampling=metodo)
    return out


def leer_si_existe(ruta):
    if not ruta.exists():
        return None
    with rasterio.open(ruta) as f:
        return f.read(1)


dem_bueno = lambda d: d is not None and bool(np.isfinite(d).all() and -500 < d.min() < d.max() < 9000)
wc_bueno = lambda w: w is not None and (w > 0).mean() >= 0.5

In [ ]:
GRILLAS = {}
for ciudad, (o, s, e, n) in CAJAS.items():
    xs, ys = Transformer.from_crs("EPSG:4326", f"EPSG:{EPSG[ciudad]}", always_xy=True).transform([o, e, o, e], [s, s, n, n])
    x0, x1, y0, y1 = 5 * np.floor(min(xs) / 5), 5 * np.ceil(max(xs) / 5), 5 * np.floor(min(ys) / 5), 5 * np.ceil(max(ys) / 5)
    GRILLAS[ciudad] = dict(driver="GTiff", height=int((y1 - y0) / 5), width=int((x1 - x0) / 5), count=1, dtype="float32",
                           crs=f"EPSG:{EPSG[ciudad]}", transform=from_origin(x0, y1, 5, 5), compress="deflate", predictor=3,
                           tiled=True, nodata=np.nan, BIGTIFF="IF_SAFER")
    carpeta = Path(f"Data/procesado/nisar/{ciudad}")
    carpeta.mkdir(parents=True, exist_ok=True)
    if not dem_bueno(leer_si_existe(carpeta / "dem.tif")):
        dem = en_grilla("cop-dem-glo-30", ciudad, Resampling.bilinear).astype("float32") + geoide((o + e) / 2, (s + n) / 2)
        if not dem_bueno(dem):
            raise RuntimeError(f"DEM de {ciudad} inválido: {np.nanmin(dem)} a {np.nanmax(dem)} m")
        with rasterio.open(carpeta / "dem.tif", "w", **GRILLAS[ciudad]) as f:
            f.write(dem, 1)
        del dem
    if not wc_bueno(leer_si_existe(carpeta / "worldcover.tif")):
        wc = en_grilla("esa-worldcover", ciudad, Resampling.nearest)
        if not wc_bueno(wc):
            raise RuntimeError(f"WorldCover de {ciudad} vacío: {(wc > 0).mean():.0%} con dato")
        with rasterio.open(carpeta / "worldcover.tif", "w", **{**GRILLAS[ciudad], "dtype": "uint8", "nodata": 0, "predictor": 1}) as f:
            f.write(wc, 1)
        del wc

pd.DataFrame([{"ciudad": c, "EPSG": g["crs"], "filas": g["height"], "columnas": g["width"],
               "km²": g["height"] * g["width"] * 25 / 1e6} for c, g in GRILLAS.items()]).round(1)

### 3.3 Geocodificación de las RSLC

HH calibrado (sigma0, sin el ruido equivalente) en la grilla de cada ciudad.

In [ ]:
def leer(clave):
    ciudad = clave.rsplit("_", 2)[0]
    with rasterio.open(f"Data/procesado/nisar/{ciudad}/hh_{clave}.tif") as f:
        hh, perfil = f.read(1), f.profile
    return hh, np.isfinite(hh), perfil


def suave(hh, valido, n=5):
    mediana = ndimage.median_filter(np.where(valido, hh, -40), size=n, mode="nearest")
    return np.where(valido, mediana, np.nan).astype("float32")


def etiqueta(clave):
    with rasterio.open(f"Data/procesado/etiquetas/{clave}.tif") as f:
        return f.read(1)

In [ ]:
def a_grilla(h5s, ciudad):
    p, (o, s, e, n) = GRILLAS[ciudad], CAJAS[ciudad]
    with rasterio.open(f"Data/procesado/nisar/{ciudad}/dem.tif") as f:
        dem = f.read(1)
    if not dem_bueno(dem):
        raise ValueError(f"El DEM de {ciudad} tiene NaN, inf o está fuera de rango: vuelve a correr la 3.1, que lo rehace")
    alto, ancho = dem.shape
    cc, ff = np.meshgrid(np.arange(0, ancho, 10), np.arange(0, alto, 10))
    lon, lat = Transformer.from_crs(p["crs"], "EPSG:4326", always_xy=True).transform(
        p["transform"].c + (cc + 0.5) * 5, p["transform"].f - (ff + 0.5) * 5)
    fi, fj = np.meshgrid(np.arange(alto, dtype="float32") / 10, np.arange(ancho, dtype="float32") / 10, indexing="ij")
    hh = np.full(dem.shape, np.nan, "float32")
    huella = np.zeros(dem.shape, bool)
    for h5 in h5s:
        with h5py.File(h5) as f:
            g, sw = f["science/LSAR/RSLC/metadata/geolocationGrid"], f["science/LSAR/RSLC/swaths"]
            t, r = sw["zeroDopplerTime"][()], sw["frequencyA/slantRange"][()]
            ta, ra = np.meshgrid((g["zeroDopplerTime"][()] - t[0]) / (t[1] - t[0]),
                                 (g["slantRange"][()] - r[0]) / (r[1] - r[0]), indexing="ij")
            alturas = g["heightAboveEllipsoid"][()]
            k0 = int(np.clip(np.searchsorted(alturas, dem.min(), "right") - 1, 0, len(alturas) - 2))
            k1 = int(np.clip(np.searchsorted(alturas, dem.max()), k0 + 1, len(alturas) - 1))
            capas = []
            for k in (k0, k1):
                X, Y = g["coordinateX"][k], g["coordinateY"][k]
                m = (X > o - 0.15) & (X < e + 0.15) & (Y > s - 0.15) & (Y < n + 0.15)
                grueso = LinearNDInterpolator(np.c_[X[m], Y[m]], np.c_[ta[m], ra[m]])(lon, lat).astype("float32")
                capas.append([ndimage.map_coordinates(grueso[..., i], [fi, fj], order=1, mode="nearest") for i in (0, 1)])
            w = (dem - alturas[k0]) / (alturas[k1] - alturas[k0])
            az = np.nan_to_num(capas[0][0] + w * (capas[1][0] - capas[0][0]), nan=-9)
            rg = np.nan_to_num(capas[0][1] + w * (capas[1][1] - capas[0][1]), nan=-9)
            dentro = (az >= 0) & (az <= len(t) - 1) & (rg >= 0) & (rg <= len(r) - 1)
            huella |= dentro
            if not dentro.any():
                continue
            a0, r0 = int(az[dentro].min()), int(rg[dentro].min())
            slc = sw["frequencyA/HH"][a0:int(az[dentro].max()) + 2, r0:int(rg[dentro].max()) + 2].astype("complex64")
            cal = f["science/LSAR/RSLC/metadata/calibrationInformation/geometry"]
            lut, lt, lr = cal["sigma0"][()], cal["zeroDopplerTime"][()], cal["slantRange"][()]
            ruido_g = f["science/LSAR/RSLC/metadata/calibrationInformation/frequencyA/noiseEquivalentBackscatter"]
            nes, nt, nr = ruido_g["HH"][()], ruido_g["zeroDopplerTime"][()], ruido_g["slantRange"][()]
        slc[slc == 0] = np.nan
        muestrear = lambda x: ndimage.map_coordinates(x, [az - a0, rg - r0], order=1, cval=np.nan, prefilter=False)
        ia = (t[0] + az * (t[1] - t[0]) - lt[0]) / (lt[1] - lt[0])
        ir = (r[0] + rg * (r[1] - r[0]) - lr[0]) / (lr[1] - lr[0])
        nuevo = np.isnan(hh) & dentro
        L = ndimage.map_coordinates(lut.astype("float32"), [ia[nuevo], ir[nuevo]], order=1, mode="nearest")
        ia_n = ((t[0] + az * (t[1] - t[0]) - nt[0]) / (nt[1] - nt[0]))[nuevo]
        ir_n = ((r[0] + rg * (r[1] - r[0]) - nr[0]) / (nr[1] - nr[0]))[nuevo]
        ruido = ndimage.map_coordinates(nes.astype("float32"), [ia_n, ir_n], order=1, mode="nearest")
        potencia = muestrear(np.abs(slc) ** 2)[nuevo]
        hh[nuevo] = 10 * np.log10(np.maximum(potencia - ruido, 0.1 * ruido)) - 20 * np.log10(L)
    return hh, huella


for e in escenas.itertuples():
    carpeta = Path(f"Data/procesado/nisar/{e.ciudad}")
    ruta_hh, ruta_cub = carpeta / f"hh_{e.clave}.tif", carpeta / f"cub_{e.clave}.tif"
    if ruta_hh.exists() and ruta_cub.exists():
        with rasterio.open(ruta_hh) as f:
            if (f.height, f.width) == (GRILLAS[e.ciudad]["height"], GRILLAS[e.ciudad]["width"]):
                continue
    hh, huella = a_grilla(e.h5, e.ciudad)
    with rasterio.open(ruta_hh, "w", **GRILLAS[e.ciudad]) as f:
        f.write(hh, 1)
    with rasterio.open(ruta_cub, "w", **{**GRILLAS[e.ciudad], "dtype": "uint8", "nodata": 0, "predictor": 1}) as f:
        f.write(huella.astype("uint8"), 1)
    del hh, huella
print(len(escenas), "escenas en la grilla de 5 m")

In [ ]:
ejemplo = escenas[escenas.dia == "lluvia"].groupby("ciudad").clave.last()
columnas = 6
fig, ejes = plt.subplots(int(np.ceil(len(ejemplo) / columnas)), columnas,
                         figsize=(4 * columnas, 4.6 * np.ceil(len(ejemplo) / columnas)), squeeze=False)
for ax, clave in zip(ejes.flat, ejemplo):
    hh, valido, perfil = leer(clave)
    paso = 8 if clave.startswith("lima") else 4
    ax.imshow(suave(hh, valido, 3)[::paso, ::paso], cmap="gray", vmin=-25, vmax=5)
    ax.set_title(f"{clave}\nHH (dB), cobertura {valido.mean():.0%}", fontsize=9)
for ax in ejes.flat:
    ax.axis("off")
plt.tight_layout()

### 3.4 Pendiente y agua permanente

In [ ]:
PENDIENTE, WORLDCOVER = {}, {}
for ciudad in CAJAS:
    dem = rasterio.open(f"Data/procesado/nisar/{ciudad}/dem.tif").read(1)
    dy, dx = np.gradient(ndimage.uniform_filter(dem, 7), 5)
    PENDIENTE[ciudad] = np.degrees(np.arctan(np.hypot(dx, dy)))
    WORLDCOVER[ciudad] = rasterio.open(f"Data/procesado/nisar/{ciudad}/worldcover.tif").read(1)
LEJOS = {c: ndimage.maximum_filter((w == 80).astype("uint8"), 81) == 0 for c, w in WORLDCOVER.items()}

columnas = 6
filas = int(np.ceil(len(CAJAS) / columnas))
fig, ejes = plt.subplots(2 * filas, columnas, figsize=(4 * columnas, 8 * filas), squeeze=False)
for j, ciudad in enumerate(CAJAS):
    f, k = 2 * (j // columnas), j % columnas
    paso = 8 if ciudad == "lima" else 4
    ejes[f, k].imshow(PENDIENTE[ciudad][::paso, ::paso], cmap="magma", vmin=0, vmax=30)
    ejes[f, k].set_title(f"{ciudad}: pendiente (°)", fontsize=9)
    ejes[f + 1, k].imshow(WORLDCOVER[ciudad][::paso, ::paso] == 80, cmap="Blues")
    ejes[f + 1, k].set_title(f"{ciudad}: agua permanente ({(WORLDCOVER[ciudad] == 80).mean():.1%})", fontsize=9)
for ax in ejes.flat:
    ax.axis("off")
plt.tight_layout()

### 3.5 Etiqueta

Umbral de HH (5 × 5) que mejor separa el agua permanente de WorldCover de la tierra en terreno plano (< 5°). En los días
secos la etiqueta es solo el agua permanente. Se borran las manchas de menos de 40 píxeles.

In [ ]:
def umbral_wc(s, valido, plano, wc):
    agua = plano & valido & (wc == 80)
    tierra = plano & valido & np.isin(wc, [10, 20, 30, 40, 50, 60, 70, 100])
    if agua.sum() < 1000 or tierra.sum() < 1000:
        return np.nan
    rng = np.random.default_rng(SEMILLA)
    a = s[agua][rng.choice(agua.sum(), 200_000)]
    b = s[tierra][rng.choice(tierra.sum(), 200_000)]
    candidatos = np.arange(-30, -9.9, 0.25)
    equilibrio = [0.5 * ((a < c).mean() + (b >= c).mean()) for c in candidatos]
    return float(candidatos[int(np.argmax(equilibrio))])


Path("Data/procesado/etiquetas").mkdir(parents=True, exist_ok=True)
UMBRALES, SIN_UMBRAL = [], []
COLA = list(escenas.itertuples())
for e in COLA:
    hh, valido, perfil = leer(e.clave)
    s = suave(hh, valido)
    wc = WORLDCOVER[e.ciudad]
    plano = valido & (PENDIENTE[e.ciudad] < 5)
    if e.dia == "seco":
        umbral, origen = np.nan, "día seco"
    elif e.clave in SIN_UMBRAL:
        umbral = float(np.median([u["umbral_HH_dB"] for u in UMBRALES if u["origen"] == "WorldCover"]))
        origen = "mediana de las otras lluvias"
    else:
        umbral, origen = umbral_wc(s, valido, plano, wc), "WorldCover"
        if np.isnan(umbral):
            SIN_UMBRAL.append(e.clave)
            COLA.append(e)
            continue
    agua = valido & (wc == 80) if e.dia == "seco" else ndimage.binary_opening((s < umbral) & plano, iterations=2)
    manchas, _ = ndimage.label(agua)
    agua &= np.bincount(manchas.ravel())[manchas] >= 40
    perfil.update(dtype="uint8", nodata=255, predictor=1)
    with rasterio.open(f"Data/procesado/etiquetas/{e.clave}.tif", "w", **perfil) as f:
        f.write(np.where(valido, agua, 255).astype("uint8"), 1)
    UMBRALES.append({"clave": e.clave, "dia": e.dia, "umbral_HH_dB": umbral, "origen": origen, "agua_%": 100 * agua[valido].mean()})
pd.DataFrame(UMBRALES).round(2)

### 3.6 Control de calidad

Una escena sirve si cobertura ≥ 0.40, AUC de HH ≥ 0.80, recall ≥ 0.60 y falsa agua ≤ 0.15.

In [ ]:
FILAS = []
for e in escenas[escenas.dia == "lluvia"].itertuples():
    hh, valido, perfil = leer(e.clave)
    s, et, wc = suave(hh, valido), etiqueta(e.clave), WORLDCOVER[e.ciudad]
    agua = np.flatnonzero((et != 255) & (wc == 80))
    tierra = np.flatnonzero((et != 255) & np.isin(wc, [10, 20, 30, 40, 50, 60, 70, 100]))
    fila = {"clave": e.clave, "ciudad": e.ciudad, "rol": e.rol, "cobertura": valido.sum() / rasterio.open(f"Data/procesado/nisar/{e.ciudad}/cub_{e.clave}.tif").read(1).astype(bool).sum(), "pixeles_agua_worldcover": len(agua)}
    if len(agua) >= 1000:
        a, t = [np.random.default_rng(SEMILLA).choice(x, 100_000) for x in (agua, tierra)]
        fila |= {"auc_hh": mannwhitneyu(s.flat[t], s.flat[a]).statistic / (len(a) * len(t)),
                 "recall_mascara": (et.flat[agua] == 1).mean(), "falsa_agua": (et.flat[tierra] == 1).mean()}
    FILAS.append(fila)
DATOS = pd.DataFrame(FILAS)
DATOS["sirve"] = ((DATOS.cobertura >= 0.4) & (DATOS.auc_hh >= 0.8) & (DATOS.recall_mascara >= 0.6) & (DATOS.falsa_agua <= 0.15))
DATOS.to_csv("Data/procesado/validacion_datos.csv", index=False)
DATOS.round(3)

In [ ]:
fig, ejes = plt.subplots(int(np.ceil(len(DATOS) / 4)), 4, figsize=(16, 4.6 * np.ceil(len(DATOS) / 4)), squeeze=False)
for ax, e in zip(ejes.flat, DATOS.itertuples()):
    hh, valido, perfil = leer(e.clave)
    paso = 12 if e.ciudad == "lima" else 6
    et = etiqueta(e.clave)[::paso, ::paso]
    ax.imshow(suave(hh, valido, 3)[::paso, ::paso], cmap="gray", vmin=-25, vmax=5)
    ax.imshow(np.ma.masked_where(et != 1, et), cmap="cool", vmin=0, vmax=1)
    ax.contour(WORLDCOVER[e.ciudad][::paso, ::paso] == 80, [0.5], colors="yellow", linewidths=0.5)
    ax.set_title(f"{e.clave}: {'sirve' if e.sirve else 'no sirve'}", fontsize=9, color="green" if e.sirve else "red")
for ax in ejes.flat:
    ax.axis("off")
plt.suptitle("máscara del mismo día (rosa) y agua permanente de WorldCover (amarillo)")
plt.tight_layout()

### 3.7 Parches de 256 × 256

In [ ]:
NEGATIVOS = escenas[escenas.dia == "seco"][["clave", "rol"]]
USAR = pd.concat([DATOS[DATOS.sirve & DATOS.rol.isin(["train", "val"])][["clave", "rol"]], NEGATIVOS], ignore_index=True)
X, Y, M, INFO = [], [], [], []
for e in USAR.itertuples():
    hh, valido, perfil = leer(e.clave)
    s, et = suave(hh, valido), etiqueta(e.clave)
    for i in range(0, et.shape[0] - 255, 256):
        for j in range(0, et.shape[1] - 255, 256):
            p = et[i:i + 256, j:j + 256]
            if (p != 255).mean() < 0.5:
                continue
            X.append(s[None, i:i + 256, j:j + 256].astype("float16"))
            Y.append(p == 1)
            M.append(p != 255)
            INFO.append({"clave": e.clave, "rol": e.rol, "agua": (p == 1).mean(), "fila": i, "col": j})
X, Y, M, INFO = np.array(X), np.array(Y), np.array(M), pd.DataFrame(INFO)
rng = np.random.default_rng(SEMILLA)
mantener = np.ones(len(INFO), bool)
for clave, g in INFO.groupby("clave"):
    if len(g) > 400:
        mantener[rng.choice(g.index.to_numpy(), len(g) - 400, replace=False)] = False
X, Y, M, INFO = X[mantener], Y[mantener], M[mantener], INFO[mantener].reset_index(drop=True)
TR, VA = (INFO.rol == "train").to_numpy(), (INFO.rol == "val").to_numpy()
INFO.groupby(["rol", "clave"]).agg(parches=("agua", "size"), agua_media=("agua", "mean")).round(3)

In [ ]:
ejemplos = INFO[INFO.agua.between(0.05, 0.5)].groupby("clave").head(2).index[:6]
fig, ejes = plt.subplots(2, len(ejemplos), figsize=(3 * len(ejemplos), 6.2), squeeze=False)
for c, k in enumerate(ejemplos):
    ejes[0, c].imshow(X[k, 0].astype("float32"), cmap="gray", vmin=-25, vmax=5)
    ejes[0, c].set_title(f"{INFO.clave[k]}\nHH (dB)", fontsize=8)
    ejes[1, c].imshow(np.where(M[k], Y[k], np.nan), cmap="Blues", vmin=0, vmax=1)
    ejes[1, c].set_title("etiqueta", fontsize=8)
for ax in ejes.flat:
    ax.axis("off")

## 4. Normalización

HH recortado a [−35, 10] dB y estandarizado con la media y la desviación de train.

In [ ]:
ENTRENO = np.clip(X[TR].astype("float32"), -35, 10)
MEDIA = np.nanmean(ENTRENO, axis=(0, 2, 3))
DESV = np.nanstd(ENTRENO, axis=(0, 2, 3))
del ENTRENO
RUTA_PREVIA = Path("Data/procesado/dl/unet_resnet34.pt")
if not ENTRENAR and RUTA_PREVIA.exists():
    PREVIO = torch.load(RUTA_PREVIA)
    MEDIA, DESV = np.array(PREVIO["media"], "float32"), np.array(PREVIO["desv"], "float32")


def normalizar(x):
    return np.nan_to_num((np.clip(x.astype("float32"), -35, 10) - MEDIA[:, None, None]) / DESV[:, None, None])


muestra = normalizar(X[TR][:200])
pd.DataFrame({"media dB": MEDIA, "desviación dB": DESV, "media normalizada": muestra.mean(axis=(0, 2, 3)),
              "desviación normalizada": muestra.std(axis=(0, 2, 3))}, index=["HH"]).round(3)

## 5. U-Net

| Modelo | Codificador | Pesos iniciales |
|---|---|---|
| resnet34 | ResNet34 | ImageNet |
| efficientnet-b0 | EfficientNet-B0 | ImageNet |
| mobilenet_v2 | MobileNetV2 | ImageNet |
| resnet34_desde_cero | ResNet34 | al azar |

Pérdida focal + Dice en los píxeles evaluables, AdamW (lr 1e-4), giros de 90° y ±3 dB al azar y hasta 15 épocas por U-Net.
Parada temprana: si la pérdida de validación no mejora en 3 épocas seguidas se detiene y se queda con la mejor época. Umbral
de mejor IoU en validación.

In [ ]:
GPU = "cuda" if torch.cuda.is_available() else "cpu"
AMP = torch.bfloat16 if GPU == "cpu" or torch.cuda.get_device_capability()[0] >= 8 else torch.float16
Xtr = torch.tensor(normalizar(X[TR]), dtype=torch.float16, device=GPU)
Ytr = torch.tensor(Y[TR][:, None], device=GPU)
Mtr = torch.tensor(M[TR][:, None], device=GPU)
Xva = torch.tensor(normalizar(X[VA]), dtype=torch.float16, device=GPU)
Yva = torch.tensor(Y[VA][:, None], device=GPU)
Mva = torch.tensor(M[VA][:, None], device=GPU)


def perdida(logits, y, m):
    y, m = y.float(), m.float()
    p = torch.sigmoid(logits)
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="none")
    pt = p * y + (1 - p) * (1 - y)
    focal = ((1 - pt) ** 2 * bce * m).sum() / m.sum()
    pm = p * m
    dice = 1 - (2 * (pm * y).sum() + 1) / (pm.sum() + (y * m).sum() + 1)
    return focal + dice


def predecir(modelo, x):
    modelo.eval()
    with torch.no_grad(), torch.autocast(GPU, dtype=AMP):
        return torch.cat([torch.sigmoid(modelo(x[i:i + 32].float())).float() for i in range(0, len(x), 32)])


def metricas(p, y, m, umbral):
    a = p >= umbral
    tp, fp, fn = (a & y & m).sum().item(), (a & ~y & m).sum().item(), (~a & y & m).sum().item()
    return {"IoU": tp / max(tp + fp + fn, 1), "F1": 2 * tp / max(2 * tp + fp + fn, 1),
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1)}


print(GPU, "·", str(AMP).replace("torch.", ""), "· train", tuple(Xtr.shape), "· val", tuple(Xva.shape))

In [ ]:
Path("Data/procesado/dl").mkdir(parents=True, exist_ok=True)
MODELOS_DL = [("resnet34", "resnet34", "imagenet"), ("efficientnet-b0", "efficientnet-b0", "imagenet"),
              ("mobilenet_v2", "mobilenet_v2", "imagenet"), ("resnet34_desde_cero", "resnet34", None)]
EPOCAS = 1 if PRUEBA_RAPIDA else 15
HISTORIAL, RESULTADOS = [], []
for nombre, encoder, iniciales in MODELOS_DL:
    inicio = time.time()
    ruta_pt = Path(f"Data/procesado/dl/unet_{nombre}.pt")
    if not ENTRENAR and ruta_pt.exists():
        guardado = torch.load(ruta_pt)
        modelo = smp.Unet(encoder, encoder_weights=None, in_channels=1, classes=1).to(GPU)
        modelo.load_state_dict(guardado["pesos"])
        p = predecir(modelo, Xva)
        RESULTADOS.append({"encoder": nombre, "mejor_epoca": guardado.get("mejor_epoca", np.nan), "umbral": guardado["umbral"],
                           "segundos": time.time() - inicio, **metricas(p, Yva, Mva, guardado["umbral"])})
        continue
    torch.manual_seed(SEMILLA)
    np.random.seed(SEMILLA)
    modelo = smp.Unet(encoder, encoder_weights=iniciales, in_channels=1, classes=1).to(GPU)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-4)
    planificador = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode="min", factor=0.5, patience=2)
    escalador = torch.amp.GradScaler(GPU, enabled=AMP == torch.float16)
    mejor_perdida, sin_mejora, pesos = float("inf"), 0, None
    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        perdidas = []
        for lote in torch.randperm(len(Xtr), device=GPU).split(16):
            giro = np.random.randint(4)
            x, y, m = [t[lote].rot90(giro, [2, 3]) for t in (Xtr, Ytr, Mtr)]
            x = x.float() + (torch.rand(len(x), 1, 1, 1, device=GPU) * 6 - 3) / float(DESV[0])
            with torch.autocast(GPU, dtype=AMP):
                logits = modelo(x)
            loss = perdida(logits.float(), y, m)
            optimizador.zero_grad()
            escalador.scale(loss).backward()
            escalador.step(optimizador)
            escalador.update()
            perdidas.append(loss.item())
        p = predecir(modelo, Xva)
        iou = metricas(p, Yva, Mva, 0.5)["IoU"]
        perd_val = perdida(torch.logit(p.clamp(1e-6, 1 - 1e-6)), Yva, Mva).item()
        planificador.step(perd_val)
        HISTORIAL.append({"encoder": nombre, "epoca": epoca, "perdida_train": np.mean(perdidas),
                          "perdida_val": perd_val, "iou_val": iou})
        print(f"{nombre} · época {epoca} · pérdida train {np.mean(perdidas):.3f} · pérdida val {perd_val:.3f} · IoU val {iou:.3f}")
        if perd_val < mejor_perdida - 1e-4:
            mejor_perdida, mejor_epoca, sin_mejora = perd_val, epoca, 0
            pesos = {c: v.clone() for c, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora == 3:
                break
    modelo.load_state_dict(pesos)
    p = predecir(modelo, Xva)
    umbral = float(max(np.arange(0.1, 0.95, 0.05).round(2), key=lambda u: metricas(p, Yva, Mva, u)["IoU"]))
    torch.save({"pesos": pesos, "encoder": encoder, "nombre": nombre, "mejor_epoca": mejor_epoca, "media": MEDIA.tolist(),
                "desv": DESV.tolist(), "umbral": umbral}, ruta_pt)
    RESULTADOS.append({"encoder": nombre, "mejor_epoca": mejor_epoca, "umbral": umbral, "segundos": time.time() - inicio,
                       **metricas(p, Yva, Mva, umbral)})
    del modelo, optimizador, pesos
    torch.cuda.empty_cache()
if HISTORIAL:
    HISTORIAL = pd.DataFrame(HISTORIAL)
    HISTORIAL.to_csv("Data/procesado/dl/historial.csv", index=False)
elif Path("Data/procesado/dl/historial.csv").exists():
    HISTORIAL = pd.read_csv("Data/procesado/dl/historial.csv")
else:
    HISTORIAL = pd.DataFrame(columns=["encoder", "epoca", "perdida_train", "perdida_val", "iou_val"])

## 6. Validación

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
for encoder, h in HISTORIAL.groupby("encoder"):
    linea, = ejes[0].plot(h.epoca, h.perdida_train, "--")
    ejes[0].plot(h.epoca, h.perdida_val, color=linea.get_color(), label=encoder)
    ejes[1].plot(h.epoca, h.iou_val, "o-", label=encoder)
ejes[0].set_title("pérdida (-- train, — validación)")
ejes[1].set_title("IoU de validación (umbral 0.5)")
for ax in ejes:
    ax.set_xlabel("época")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()

In [ ]:
RESULTADOS = pd.DataFrame(RESULTADOS).sort_values("IoU", ascending=False)
RESULTADOS.to_csv("Data/procesado/dl/resultados_validacion.csv", index=False)
MEJOR = RESULTADOS.encoder.iloc[0]
RESULTADOS.round(3)

In [ ]:
GUARDADO = torch.load(f"Data/procesado/dl/unet_{MEJOR}.pt")
MODELO = smp.Unet(GUARDADO["encoder"], encoder_weights=None, in_channels=1, classes=1).to(GPU)
MODELO.load_state_dict(GUARDADO["pesos"])
UMBRAL = GUARDADO["umbral"]
p = predecir(MODELO, Xva)
INFO_VA = INFO[VA].reset_index(drop=True)
ejemplos = INFO_VA[INFO_VA.agua.between(0.05, 0.5)].head(5).index
fig, ejes = plt.subplots(len(ejemplos), 4, figsize=(12, 3 * len(ejemplos)), squeeze=False)
for fila, k in zip(ejes, ejemplos):
    fila[0].imshow(X[VA][k, 0].astype("float32"), cmap="gray", vmin=-25, vmax=5)
    fila[0].set_title(f"{INFO_VA.clave[k]} · HH", fontsize=8)
    fila[1].imshow(Yva[k, 0].cpu(), cmap="Blues")
    fila[1].set_title("etiqueta", fontsize=8)
    fila[2].imshow(p[k, 0].cpu(), cmap="magma", vmin=0, vmax=1)
    fila[2].set_title("probabilidad U-Net", fontsize=8)
    fila[3].imshow((p[k, 0] >= UMBRAL).cpu(), cmap="Blues")
    fila[3].set_title(f"máscara (umbral {UMBRAL})", fontsize=8)
for ax in ejes.flat:
    ax.axis("off")
plt.tight_layout()

## 7. Machine learning por píxel

Cada píxel evaluable de los parches de train y validación es una fila:

| Variable | Qué es |
|---|---|
| `hh` | HH (dB) suavizado 5 × 5 |
| `hh_med3` | mediana 3 × 3 |
| `hh_media5` | media 5 × 5 |
| `hh_std5` | desviación 5 × 5 |
| `dem` | elevación (m) |
| `pendiente` | pendiente del terreno (°) |

### 7.1 Tabla de píxeles

In [ ]:
VARIABLES = ["hh", "hh_med3", "hh_media5", "hh_std5", "dem", "pendiente"]
N_PIXELES = {"train": 20_000, "val": 30_000} if PRUEBA_RAPIDA else {"train": 120_000, "val": 150_000}
rng = np.random.default_rng(SEMILLA)
validos = M.reshape(len(M), -1).sum(1)
FUENTES_DEM = {c: rasterio.open(f"Data/procesado/nisar/{c}/dem.tif") for c in PENDIENTE}
TABLAS = {}
for conjunto, parches in [("train", TR), ("val", VA)]:
    idx = np.flatnonzero(parches)
    elegidos, cuantos = np.unique(rng.choice(idx, N_PIXELES[conjunto], p=validos[idx] / validos[idx].sum()), return_counts=True)
    partes = []
    for k, n in zip(elegidos, cuantos):
        pos = rng.choice(np.flatnonzero(M[k].ravel()), n)
        x = X[k, 0].astype("float32")
        x = np.where(np.isfinite(x), x, np.nanmedian(x))
        m5 = ndimage.uniform_filter(x, 5)
        i, j, ciudad = INFO.fila[k], INFO.col[k], INFO.clave[k].rsplit("_", 2)[0]
        capas = [x, ndimage.median_filter(x, 3), m5, np.sqrt(np.maximum(ndimage.uniform_filter(x ** 2, 5) - m5 ** 2, 0)),
                 FUENTES_DEM[ciudad].read(1, window=Window(j, i, 256, 256)).astype("float32"),
                 PENDIENTE[ciudad][i:i + 256, j:j + 256]]
        partes.append(pd.DataFrame({v: capa.ravel()[pos] for v, capa in zip(VARIABLES, capas)}
                                   | {"agua": Y[k].ravel()[pos].astype(int), "parche": k, "pos": pos}))
    TABLAS[conjunto] = pd.concat(partes, ignore_index=True).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
for fuente in FUENTES_DEM.values():
    fuente.close()
pd.DataFrame({c: {"píxeles": len(t), "parches": t.parche.nunique(), "agua %": 100 * t.agua.mean()} for c, t in TABLAS.items()}).T.round(2)

In [ ]:
TABLAS["train"].head()

### 7.2 Selección de variables

Con train: relevancia por Mann-Whitney (p < 0.05 y |AUC − 0.5| ≥ 0.05) y redundancia por Spearman (|ρ| > 0.9).

In [ ]:
x_tr, y_tr = TABLAS["train"][VARIABLES], TABLAS["train"].agua.to_numpy()
x_va, y_va = TABLAS["val"][VARIABLES], TABLAS["val"].agua.to_numpy()
SELECCION = pd.DataFrame({"AUC": [roc_auc_score(y_tr, x_tr[v]) for v in VARIABLES],
                          "p_mann_whitney": [mannwhitneyu(x_tr[v][y_tr == 1], x_tr[v][y_tr == 0]).pvalue for v in VARIABLES]},
                         index=VARIABLES)
SELECCION["relevancia"] = (SELECCION.AUC - 0.5).abs()
RHO = pd.DataFrame(np.abs(spearmanr(x_tr)[0]), index=VARIABLES, columns=VARIABLES)
ELEGIDAS = []
for v in SELECCION.sort_values("relevancia", ascending=False).index:
    if SELECCION.p_mann_whitney[v] < 0.05 and SELECCION.relevancia[v] >= 0.05 and (RHO.loc[v, ELEGIDAS] <= 0.9).all():
        ELEGIDAS.append(v)
ELEGIDAS = ELEGIDAS or ["hh"]
SELECCION["elegida"] = SELECCION.index.isin(ELEGIDAS)
SELECCION.join(RHO.add_prefix("rho_")).round(3)

### 7.3 Modelos

En cada celda: crear → entrenar con train → evaluar en validación con el umbral de mejor IoU. La regresión logística es el
modelo base. HistGradientBoosting, XGBoost y la red MLP tienen parada temprana: se detienen cuando dejan de mejorar en
validación (10 iteraciones, 10 rondas y 5 épocas, respectivamente).

In [ ]:
UMBRALES_PRUEBA = np.arange(0.05, 0.96, 0.05).round(2)
PROB_VA, AJUSTADOS, COLUMNAS, FILAS_ML = {}, {}, {}, []


def evaluar(nombre, modelo, inicio):
    segundos = time.time() - inicio
    prob = modelo.predict_proba(x_va[ELEGIDAS])[:, 1]
    umbral = float(max(UMBRALES_PRUEBA, key=lambda u: jaccard_score(y_va, (prob >= u).astype(int))))
    pred = (prob >= umbral).astype(int)
    PROB_VA[nombre], AJUSTADOS[nombre], COLUMNAS[nombre] = ("ML", prob, umbral, segundos), modelo, ELEGIDAS
    FILAS_ML.append(pd.DataFrame([{"modelo": nombre, "umbral": umbral, "IoU": jaccard_score(y_va, pred), "F1": f1_score(y_va, pred),
                                   "Precision": precision_score(y_va, pred, zero_division=0), "Recall": recall_score(y_va, pred),
                                   "Accuracy": accuracy_score(y_va, pred), "AUC": roc_auc_score(y_va, prob), "segundos": segundos}]))
    return FILAS_ML[-1].round(3)

In [ ]:
inicio = time.time()
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEMILLA)).fit(x_tr[ELEGIDAS], y_tr)
evaluar("Regresión logística (base)", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = GaussianNB().fit(x_tr[ELEGIDAS], y_tr)
evaluar("Naive Bayes", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = DecisionTreeClassifier(max_depth=12, random_state=SEMILLA).fit(x_tr[ELEGIDAS], y_tr)
evaluar("Árbol de decisión", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)).fit(x_tr[ELEGIDAS], y_tr)
evaluar("KNN", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, probability=True, random_state=SEMILLA)).fit(
    x_tr[ELEGIDAS].iloc[:8_000], y_tr[:8_000])
evaluar("SVM rbf (8 000 píxeles)", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=SEMILLA).fit(x_tr[ELEGIDAS], y_tr)
evaluar("Random Forest", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = AdaBoostClassifier(n_estimators=100, random_state=SEMILLA).fit(x_tr[ELEGIDAS], y_tr)
evaluar("AdaBoost", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = HistGradientBoostingClassifier(max_iter=100, early_stopping=True, n_iter_no_change=10, validation_fraction=0.1,
                                        random_state=SEMILLA).fit(x_tr[ELEGIDAS], y_tr)
evaluar("HistGradientBoosting", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, early_stopping_rounds=10, eval_metric="logloss", n_jobs=-1,
                       random_state=SEMILLA).fit(x_tr[ELEGIDAS], y_tr, eval_set=[(x_va[ELEGIDAS], y_va)], verbose=False)
evaluar("XGBoost", modelo, inicio)

In [ ]:
inicio = time.time()
modelo = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=60, early_stopping=True,
                                                       n_iter_no_change=5, random_state=SEMILLA)).fit(x_tr[ELEGIDAS], y_tr)
FILA_MLP = evaluar("Red neuronal (MLP)", modelo, inicio)
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.2))
ejes[0].plot(np.arange(1, len(modelo[-1].loss_curve_) + 1), modelo[-1].loss_curve_, color="#2a78d6", lw=2)
ejes[0].set_title("MLP: pérdida de entrenamiento (log-loss)")
ejes[1].plot(np.arange(1, len(modelo[-1].validation_scores_) + 1), modelo[-1].validation_scores_, color="#2a78d6", lw=2)
ejes[1].set_title("MLP: accuracy en su 10 % de validación interna")
for ax in ejes:
    ax.set_xlabel("época")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
FILA_MLP

### 7.4 Optimización del ganador (GridSearchCV)

El modelo con mejor IoU en validación se optimiza con su propia grilla: `GroupKFold(3)` por parche e IoU como criterio. Si el
mejor valor queda en el borde, la grilla se amplía.

In [ ]:
RESUMEN_ML = pd.concat(FILAS_ML).sort_values("IoU", ascending=False).reset_index(drop=True)
GANADOR_ML = RESUMEN_ML.modelo.iloc[0]
RESUMEN_ML.round(3)

In [ ]:
REJILLAS = {
    "Regresión logística (base)": {"logisticregression__C": [0.1, 1.0, 10.0]},
    "Naive Bayes": {"var_smoothing": [1e-9, 1e-7, 1e-5]},
    "Árbol de decisión": {"max_depth": [8, 12, 16], "min_samples_leaf": [1, 5, 20]},
    "KNN": {"kneighborsclassifier__n_neighbors": [5, 15, 31]},
    "SVM rbf (8 000 píxeles)": {"svc__C": [0.3, 1.0, 3.0]},
    "Random Forest": {"n_estimators": [50, 150], "max_depth": [8, 16, None], "min_samples_leaf": [1, 5]},
    "AdaBoost": {"n_estimators": [50, 100, 200], "learning_rate": [0.5, 1.0]},
    "HistGradientBoosting": {"max_iter": [100, 200], "learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31, 63]},
    "XGBoost": {"n_estimators": [100, 200], "max_depth": [4, 6, 8], "learning_rate": [0.05, 0.1]},
    "Red neuronal (MLP)": {"mlpclassifier__alpha": [1e-4, 1e-3, 1e-2]},
}
MINIMOS = {"n_estimators": 10, "max_depth": 2, "min_samples_leaf": 1, "n_neighbors": 1, "max_iter": 10, "max_leaf_nodes": 2}
REJILLA = REJILLAS[GANADOR_ML]
PROBADOS = {p: set(v) for p, v in REJILLA.items()}
AJUSTE = {"eval_set": [(x_va[ELEGIDAS], y_va)], "verbose": False} if GANADOR_ML == "XGBoost" else {}
n = min(8_000 if GANADOR_ML.startswith("SVM") else 20_000, len(y_tr))
for _ in range(4):
    BUSQUEDA = GridSearchCV(clone(AJUSTADOS[GANADOR_ML]), REJILLA, cv=GroupKFold(3), scoring="jaccard").fit(
        x_tr[ELEGIDAS].iloc[:n], y_tr[:n], groups=TABLAS["train"].parche.iloc[:n], **AJUSTE)
    NUMEROS = {p: [v for v in PROBADOS[p] if v is not None] for p in PROBADOS}
    arriba = [p for p, v in BUSQUEDA.best_params_.items() if v is not None and None not in PROBADOS[p] and v == max(NUMEROS[p])]
    abajo = [p for p, v in BUSQUEDA.best_params_.items()
             if v is not None and v == min(NUMEROS[p]) and v > MINIMOS.get(p.split("__")[-1], 0)]
    if not arriba and not abajo:
        break
    REJILLA = {p: [v, 2 * v, 3 * v] if p in arriba
               else sorted({max(v // 4, MINIMOS[p.split("__")[-1]]), max(v // 2, MINIMOS[p.split("__")[-1]]), v})
               if p in abajo and isinstance(v, int) else [v / 4, v / 2, v] if p in abajo else [v]
               for p, v in BUSQUEDA.best_params_.items()}
    PROBADOS = {p: PROBADOS[p] | set(v) for p, v in REJILLA.items()}
(pd.DataFrame(BUSQUEDA.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
 .sort_values("rank_test_score").head(6).round(3))

In [ ]:
inicio = time.time()
n = 8_000 if GANADOR_ML.startswith("SVM") else len(y_tr)
modelo = clone(AJUSTADOS[GANADOR_ML]).set_params(**BUSQUEDA.best_params_).fit(x_tr[ELEGIDAS].iloc[:n], y_tr[:n], **AJUSTE)
evaluar(f"{GANADOR_ML} (optimizado)", modelo, inicio)

### 7.5 Machine learning vs U-Net en validación

In [ ]:
VA_POS = np.cumsum(VA) - 1
FILAS_GPU = torch.as_tensor(VA_POS[TABLAS["val"].parche.to_numpy()], device=GPU)
POS_GPU = torch.as_tensor(TABLAS["val"].pos.to_numpy(), device=GPU)
for nombre in RESULTADOS.encoder:
    guardado = torch.load(f"Data/procesado/dl/unet_{nombre}.pt")
    red = smp.Unet(guardado["encoder"], encoder_weights=None, in_channels=1, classes=1).to(GPU)
    red.load_state_dict(guardado["pesos"])
    PROB_VA[f"U-Net {nombre}"] = ("DL", predecir(red, Xva).flatten(1)[FILAS_GPU, POS_GPU].cpu().numpy(), guardado["umbral"],
                                  RESULTADOS.set_index("encoder").segundos[nombre])
    del red
torch.cuda.empty_cache()

FILAS = []
for nombre, (familia, prob, umbral, segundos) in PROB_VA.items():
    if umbral is None:
        umbral = float(max(UMBRALES_PRUEBA, key=lambda u: jaccard_score(y_va, (prob >= u).astype(int))))
    pred = (prob >= umbral).astype(int)
    FILAS.append({"modelo": nombre, "familia": familia, "umbral": umbral, "IoU": jaccard_score(y_va, pred),
                  "F1": f1_score(y_va, pred), "Precision": precision_score(y_va, pred, zero_division=0),
                  "Recall": recall_score(y_va, pred), "Accuracy": accuracy_score(y_va, pred), "AUC": roc_auc_score(y_va, prob),
                  "segundos": segundos})
COMPARACION_MODELOS = pd.DataFrame(FILAS).sort_values("IoU", ascending=False).reset_index(drop=True)
COMPARACION_MODELOS.to_csv("Data/procesado/dl/comparacion_ml_dl.csv", index=False)
UMBRALES_VA = COMPARACION_MODELOS.set_index("modelo").umbral
Path("Data/procesado/ml").mkdir(parents=True, exist_ok=True)
for nombre, modelo in AJUSTADOS.items():
    archivo = re.sub("[^a-z0-9]+", "_", unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode().lower()).strip("_")
    joblib.dump({"modelo": modelo, "columnas": COLUMNAS[nombre], "umbral": float(UMBRALES_VA[nombre])},
                f"Data/procesado/ml/ml_{archivo}.joblib", compress=3)

fig, ax = plt.subplots(figsize=(9, 0.4 * len(COMPARACION_MODELOS) + 1.2))
orden = COMPARACION_MODELOS[::-1]
ax.barh(orden.modelo, orden.IoU, height=0.6, color=orden.familia.map({"ML": "#2a78d6", "DL": "#eb6834"}).tolist())
ax.legend([plt.Rectangle((0, 0), 1, 1, color="#2a78d6"), plt.Rectangle((0, 0), 1, 1, color="#eb6834")],
          ["machine learning por píxel", "U-Net"], loc="lower right", frameon=False)
ax.set_xlabel(f"IoU en validación ({len(y_va):,} píxeles)")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
COMPARACION_MODELOS.round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
REPORTES = []
for ax, familia in zip(ejes, ["ML", "DL"]):
    fila = COMPARACION_MODELOS[COMPARACION_MODELOS.familia == familia].iloc[0]
    pred = (PROB_VA[fila.modelo][1] >= fila.umbral).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y_va, pred), display_labels=["tierra", "agua"]).plot(
        ax=ax, cmap="Blues", values_format=",", colorbar=False)
    ax.set_title(f"{fila.modelo} (umbral {fila.umbral})", fontsize=10)
    REPORTES.append(pd.DataFrame(classification_report(y_va, pred, target_names=["tierra", "agua"], output_dict=True)).T
                    .drop(index="accuracy").assign(modelo=fila.modelo))
plt.tight_layout()
plt.show()
pd.concat(REPORTES).set_index("modelo", append=True).swaplevel().round(3)

## 8. Predicción y prueba externa

Cada escena se predice con ventanas de 256 píxeles y paso de 128. La prueba externa son las lluvias de rol `test` y las
garúas de Lima (`garua`) que pasan el control de calidad.

In [ ]:
del Xtr, Ytr, Mtr, Xva, Yva, Mva, X, Y, M
torch.cuda.empty_cache()


def mapa(modelo, hh):
    modelo.eval()
    x = normalizar(hh[None])
    alto, ancho = hh.shape
    g = np.exp(-(np.arange(256) - 127.5) ** 2 / (2 * 32.0 ** 2)).astype("float32")
    peso = np.outer(g, g)
    suma, veces = np.zeros((alto, ancho), "float32"), np.zeros((alto, ancho), "float32")
    filas = list(range(0, alto - 256, 128)) + [alto - 256]
    columnas = list(range(0, ancho - 256, 128)) + [ancho - 256]
    with torch.no_grad(), torch.autocast(GPU, dtype=AMP):
        for i in filas:
            lote = torch.tensor(np.stack([x[:, i:i + 256, j:j + 256] for j in columnas]), dtype=torch.float32, device=GPU)
            prob = torch.sigmoid(modelo(lote)).float().cpu().numpy()[:, 0]
            for j, pj in zip(columnas, prob):
                suma[i:i + 256, j:j + 256] += pj * peso
                veces[i:i + 256, j:j + 256] += peso
    return suma / veces


def limpiar(mascara, minimo=40):
    mascara = ndimage.binary_opening(mascara, iterations=1)
    manchas, _ = ndimage.label(mascara)
    return mascara & (np.bincount(manchas.ravel())[manchas] >= minimo)


Path("Data/procesado/dl/predicciones").mkdir(parents=True, exist_ok=True)
for e in escenas.itertuples():
    hh, valido, perfil = leer(e.clave)
    p = np.where(valido, mapa(MODELO, suave(hh, valido)), np.nan)
    perfil.update(dtype="uint8", nodata=255, predictor=1)
    with rasterio.open(f"Data/procesado/dl/predicciones/{e.clave}.tif", "w", **perfil) as f:
        f.write(np.where(valido, np.round(np.nan_to_num(p) * 100), 255).astype("uint8"), 1)
del hh, p
torch.cuda.empty_cache()

In [ ]:
def metricas_np(pred, ref):
    tp, fp, fn = (pred & ref).sum(), (pred & ~ref).sum(), (~pred & ref).sum()
    return {"IoU": tp / max(tp + fp + fn, 1), "F1": 2 * tp / max(2 * tp + fp + fn, 1),
            "Precision": tp / max(tp + fp, 1), "Recall": tp / max(tp + fn, 1)}


FILAS = []
for c, rol in DATOS.loc[(DATOS.rol == "val") | (DATOS.sirve & DATOS.rol.isin(["test", "garua"])), ["clave", "rol"]].itertuples(index=False):
    ciudad = c.rsplit("_", 2)[0]
    et, wc = etiqueta(c), WORLDCOVER[ciudad]
    with rasterio.open(f"Data/procesado/dl/predicciones/{c}.tif") as f:
        p = f.read(1).astype("float32") / 100
    valido = et != 255
    zonas = [("todo", valido), ("agua permanente", valido & (wc == 80)), ("tierra a más de 200 m", valido & LEJOS[ciudad])]
    limpio = limpiar(p >= UMBRAL)
    for zona, m in zonas:
        q = p[m] >= UMBRAL
        FILAS.append({"clave": c, "rol": rol, "zona": zona, "pixeles_km2": m.sum() * 25 / 1e6, "agua_etiqueta_%": 100 * (et[m] == 1).mean(),
                      "agua_unet_%": 100 * q.mean(), **metricas_np(q, et[m] == 1)})
        FILAS.append({"clave": c, "rol": rol, "zona": zona + " (limpio)", "pixeles_km2": m.sum() * 25 / 1e6,
                      "agua_etiqueta_%": 100 * (et[m] == 1).mean(), "agua_unet_%": 100 * limpio[m].mean(),
                      **metricas_np(limpio[m], et[m] == 1)})
POR_ZONA = pd.DataFrame(FILAS)
POR_ZONA.to_csv("Data/procesado/dl/resultados_por_zona.csv", index=False)
POR_ZONA.round(3)

### 8.1 Machine learning vs U-Net en la prueba externa

In [ ]:
FILAS = []
for c, rol in DATOS.loc[DATOS.sirve & DATOS.rol.isin(["test", "garua"]), ["clave", "rol"]].itertuples(index=False):
    ciudad = c.rsplit("_", 2)[0]
    hh, valido, perfil = leer(c)
    et = etiqueta(c)
    pos = np.random.default_rng(SEMILLA).choice(np.flatnonzero(et != 255), N_PIXELES["val"])
    x = suave(hh, valido)
    x = np.where(np.isfinite(x), x, np.nanmedian(x))
    m5 = ndimage.uniform_filter(x, 5)
    with rasterio.open(f"Data/procesado/nisar/{ciudad}/dem.tif") as f:
        dem = f.read(1).astype("float32")
    capas = [x, ndimage.median_filter(x, 3), m5, np.sqrt(np.maximum(ndimage.uniform_filter(x ** 2, 5) - m5 ** 2, 0)), dem,
             PENDIENTE[ciudad]]
    tabla = pd.DataFrame({v: capa.ravel()[pos] for v, capa in zip(VARIABLES, capas)})
    y = (et.ravel()[pos] == 1).astype(int)
    with rasterio.open(f"Data/procesado/dl/predicciones/{c}.tif") as f:
        PROB = {f"U-Net {MEJOR}": f.read(1).ravel()[pos] / 100}
    PROB |= {nombre: modelo.predict_proba(tabla[COLUMNAS[nombre]])[:, 1] for nombre, modelo in AJUSTADOS.items()}
    for nombre, prob in PROB.items():
        pred = (prob >= UMBRALES_VA[nombre]).astype(int)
        FILAS.append({"escena": c, "rol": rol, "modelo": nombre, "familia": "DL" if nombre.startswith("U-Net") else "ML",
                      "IoU": jaccard_score(y, pred), "F1": f1_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
                      "Recall": recall_score(y, pred), "Accuracy": accuracy_score(y, pred), "AUC": roc_auc_score(y, prob)})
    del hh, valido, et, x, m5, dem, capas
EXTERNA_ML = pd.DataFrame(FILAS, columns=["escena", "rol", "modelo", "familia", "IoU", "F1", "Precision", "Recall", "Accuracy", "AUC"])
EXTERNA_ML = EXTERNA_ML.sort_values(["rol", "escena", "IoU"], ascending=[False, True, False]).reset_index(drop=True)
EXTERNA_ML.to_csv("Data/procesado/dl/comparacion_ml_dl_externa.csv", index=False)
EXTERNA_ML.round(3)

## 9. Agua de la lluvia

Agua de la U-Net que no es agua en el día seco de la misma pista, donde HH bajó más de 3 dB y que no toca cuerpos de agua
grandes. Las garúas de Lima y los días secos son el control: deberían dar casi nada.

In [ ]:
def sin_agua_grande(m, seco, minimo_km2=0.05, borde=3):
    manchas, _ = ndimage.label(seco)
    grande = (np.bincount(manchas.ravel()) * 25 / 1e6 >= minimo_km2)[manchas] & seco
    grande = ndimage.binary_dilation(grande, iterations=borde)
    nuevas, _ = ndimage.label(m)
    tocan = np.unique(nuevas[grande & m])
    return m & ~np.isin(nuevas, tocan[tocan > 0])


def agua_unet(c):
    with rasterio.open(f"Data/procesado/dl/predicciones/{c}.tif") as f:
        p = f.read(1)
    return (p >= UMBRAL * 100) & (p != 255)


def mediana(pilas):
    return np.concatenate([np.nanmedian(np.stack([p[i:i + 500] for p in pilas]), 0)
                           for i in range(0, len(pilas[0]), 500)]).astype("float32")


def en_ventanas(dif, urbano, lado):
    ok = urbano & ~np.isnan(dif)
    suma = ndimage.uniform_filter(np.where(ok, dif, 0), lado)
    peso = ndimage.uniform_filter(ok.astype("float32"), lado)
    return np.where(peso > 0.6, suma / np.maximum(peso, 1e-6), np.nan)


NOTICIAS = pd.DataFrame([
    ("lima", "baipás Nueva Esperanza", -12.1804, -76.9435, 60, "lima_2026-11-05", "inundado y cerrado el 5 nov (El Búho)"),
    ("lima", "VES Sector 1 Grupo 1", -12.1874, -76.9471, 200, "lima_2026-11-05", "desagüe colapsado, 4-5 nov (Caretas)"),
    ("lima", "Hospital de Emergencias VES", -12.2321, -76.9348, 100, "lima_2026-11-05", "aniego en el exterior, 5 nov (Andina)"),
], columns=["ciudad", "lugar", "lat", "lon", "radio_m", "escena", "noticia"])
NOTICIAS = NOTICIAS[NOTICIAS.escena.isin(escenas.base)].reset_index(drop=True)
NOTICIAS["lado"] = 2 * (NOTICIAS.radio_m // 5) + 1
NOTICIAS["fila"], NOTICIAS["col"] = zip(*[rowcol(GRILLAS[n.ciudad]["transform"], *Transformer.from_crs(
    "EPSG:4326", GRILLAS[n.ciudad]["crs"], always_xy=True).transform(n.lon, n.lat)) for n in NOTICIAS.itertuples()])
NOTICIAS[["ciudad", "lugar", "radio_m", "escena", "noticia"]]

In [ ]:
ROLES = dict(zip(escenas.clave, escenas.rol))
RNG = np.random.default_rng(SEMILLA)
FILAS, VISTA_LLUVIA, SITIO, EN_SECOS, VISTAS = [], {}, {}, {}, {}
for (ciudad, pista), g in escenas.groupby(["ciudad", "pista"]):
    secos_g = list(g.clave[g.dia == "seco"])
    if not secos_g:
        continue
    hs = {c: suave(*leer(c)[:2]).astype("float16") for c in g.clave}
    muestra = RNG.choice(hs[secos_g[0]].size, 2_000_000)
    comun = np.logical_and.reduce([np.isfinite(hs[c].flat[muestra]) for c in g.clave])
    nivel = {c: np.median(hs[c].flat[muestra][comun].astype("float32")) for c in g.clave}
    for c in g.clave:
        hs[c] -= np.float16(nivel[c] - np.median([nivel[r] for r in secos_g]))
    total = sum(agua_unet(r).astype("uint8") for r in secos_g)
    mar = ndimage.binary_dilation(WORLDCOVER[ciudad] == 80, iterations=2)
    for c in g.clave:
        dia = "seco" if c in secos_g else "lluvia"
        base = c.rsplit("_", 1)[0]
        refs = [r for r in secos_g if r != c]
        if not refs:
            continue
        med = mediana([hs[r] for r in refs])
        dif = hs[c] - med
        agua = agua_unet(c)
        veces = total - agua.astype("uint8") if dia == "seco" else total
        seco = (veces > 0.2 * len(refs)) | (WORLDCOVER[ciudad] == 80)
        m = sin_agua_grande(limpiar(agua & ~seco & ~mar & (dif < -3)), seco)
        tierra = (etiqueta(c) == 1) & LEJOS[ciudad] if dia == "lluvia" else None
        FILAS.append({"ciudad": ciudad, "pista": pista, "escena": c, "dia": dia, "rol": ROLES[c],
                      "agua_unet_km2": agua.sum() * 25 / 1e6, "agua_de_lluvia_km2": m.sum() * 25 / 1e6,
                      "etiqueta_en_tierra_km2": tierra.sum() * 25 / 1e6 if dia == "lluvia" else np.nan,
                      "oscura_en_secos_%": 100 * (med[tierra] < -12).mean() if dia == "lluvia" else np.nan})
        if dia == "lluvia":
            paso = 10 if ciudad == "lima" else 5
            VISTA_LLUVIA[c] = (hs[c][::paso, ::paso].astype("float32"),
                               ndimage.binary_dilation(m, iterations=paso)[::paso, ::paso], m.sum() * 25 / 1e6)
        for n in NOTICIAS[NOTICIAS.escena.isin(g.base)].itertuples():
            if base != n.escena and dia == "lluvia":
                continue
            v = en_ventanas(dif, med > -10, n.lado)
            if base != n.escena:
                EN_SECOS.setdefault(n.Index, []).append(float(v[n.fila, n.col]))
                continue
            zona = np.s_[n.fila - n.lado // 2:n.fila + n.lado // 2 + 1, n.col - n.lado // 2:n.col + n.lado // 2 + 1]
            resto = v[::n.lado // 2 + 1, ::n.lado // 2 + 1]
            resto = resto[~np.isnan(resto)]
            SITIO[n.Index] = {"unet_agua_m2": int(agua[zona].sum()) * 25, "agua_de_lluvia_m2": int(m[zona].sum()) * 25,
                              "cambio_HH_dB": round(float(v[n.fila, n.col]), 2),
                              "zonas_que_cambian_menos_%": round(float((resto < v[n.fila, n.col]).mean() * 100), 1)}
            cerca = np.s_[n.fila - 180:n.fila + 180, n.col - 180:n.col + 180]
            VISTAS[n.Index] = (hs[c][cerca].astype("float32"), med[cerca], ndimage.uniform_filter(np.nan_to_num(dif[cerca]), 9))
    del hs, total
COMPARACION = pd.DataFrame(FILAS)
COMPARACION.to_csv("Data/procesado/dl/agua_de_lluvia.csv", index=False)
COMPARACION[COMPARACION.dia == "lluvia"].round(3)

In [ ]:
(COMPARACION[COMPARACION.dia == "lluvia"].groupby(["ciudad", "pista", "rol"]).agua_de_lluvia_km2
 .agg(escenas="size", media_km2="mean", max_km2="max").round(3))

In [ ]:
fig, ejes = plt.subplots(int(np.ceil(len(VISTA_LLUVIA) / 4)), 4, figsize=(17, 5 * np.ceil(len(VISTA_LLUVIA) / 4)), squeeze=False)
for ax, (c, (fondo, mancha, km2)) in zip(ejes.flat, VISTA_LLUVIA.items()):
    ax.imshow(fondo, cmap="gray", vmin=-25, vmax=5)
    ax.imshow(np.where(mancha, 1.0, np.nan), cmap=ListedColormap(["red"]), vmin=0, vmax=1)
    ax.set_title(f"{c}: {km2:.3f} km²", fontsize=9)
for ax in ejes.flat:
    ax.axis("off")
plt.suptitle("agua de lluvia (rojo, engrosada para que se vea) sobre HH de ese día")
plt.tight_layout()

### 9.1 Lugares de las noticias

In [ ]:
def dias_secos(i):
    if i not in EN_SECOS:
        return "sin otro día seco en la pista"
    return f"{np.nanmin(EN_SECOS[i]):+.2f} a {np.nanmax(EN_SECOS[i]):+.2f}"
RESULTADO = pd.DataFrame([{"lugar": n.lugar, "escena": n.escena, "noticia": n.noticia, **SITIO.get(n.Index, {}),
                           "dias_secos_dB": dias_secos(n.Index)}
                          for n in NOTICIAS.itertuples()])
RESULTADO.to_csv("Data/procesado/dl/validacion_noticias.csv", index=False)
RESULTADO

In [ ]:
casos = NOTICIAS.drop_duplicates("escena")
fig, ejes = plt.subplots(len(casos), 3, figsize=(13, 4.4 * len(casos)), squeeze=False)
for fila, n in zip(ejes, casos.itertuples()):
    hs_v, med_v, dif_v = VISTAS[n.Index]
    fila[0].imshow(hs_v, cmap="gray", vmin=-25, vmax=5)
    fila[0].set_title(f"HH {n.escena}")
    fila[1].imshow(med_v, cmap="gray", vmin=-25, vmax=5)
    fila[1].set_title("mediana de los días secos")
    im = fila[2].imshow(dif_v, cmap="RdBu_r", vmin=-4, vmax=4)
    fila[2].set_title("cambio (dB)")
    plt.colorbar(im, ax=fila[2], shrink=0.8)
    for ax in fila:
        for m in NOTICIAS[NOTICIAS.escena == n.escena].itertuples():
            ax.add_patch(plt.Circle((m.col - n.col + 180, m.fila - n.fila + 180), m.radio_m / 5, fill=False, color="lime", lw=1.5))
        ax.axis("off")
plt.suptitle("alrededor de cada lugar con noticias (1.8 km de lado; círculos verdes)")
plt.tight_layout()

## 10. Charcos

Agua de la U-Net fuera del agua permanente de WorldCover (10 m de margen), en GeoJSON, KML y CSV.

In [ ]:
CARPETA_CHARCOS = Path("Data/procesado/dl/charcos")
CARPETA_CHARCOS.mkdir(parents=True, exist_ok=True)
CHARCOS, VISTA_FINAL = [], {}
for c in DATOS.clave:
    ciudad = c.rsplit("_", 2)[0]
    permanente = WORLDCOVER[ciudad] == 80
    m = limpiar(agua_unet(c) & ~ndimage.binary_dilation(permanente, iterations=2))
    with rasterio.open(f"Data/procesado/dl/predicciones/{c}.tif") as f:
        transformada, crs = f.transform, f.crs
    manchas_m, _ = ndimage.label(m)
    tamanos = np.bincount(manchas_m.ravel())
    RASGOS = []
    for geom, valor in shapes(manchas_m.astype("int32"), mask=m, transform=transformada):
        g = {"type": "Polygon", "coordinates": [[[float(x), float(y)] for x, y in anillo]
                                                for anillo in transform_geom(crs, "EPSG:4326", geom)["coordinates"]]}
        lon, lat = np.array(g["coordinates"][0]).mean(0)
        RASGOS.append({"type": "Feature", "geometry": g,
                       "properties": {"escena": c, "charco": int(valor), "lat": round(float(lat), 6), "lon": round(float(lon), 6),
                                      "area_m2": int(tamanos[int(valor)]) * 25}})
    with open(CARPETA_CHARCOS / f"{c}.geojson", "w") as f:
        json.dump({"type": "FeatureCollection", "features": RASGOS}, f)
    marcas = "".join(
        f"<Placemark><name>charco {r['properties']['charco']}</name><description>{r['properties']['area_m2']} m²</description>"
        "<styleUrl>#charco</styleUrl><Polygon>"
        + "".join(f"<{'outer' if k == 0 else 'inner'}BoundaryIs><LinearRing><coordinates>"
                  + " ".join(f"{x:.6f},{y:.6f},0" for x, y in anillo)
                  + f"</coordinates></LinearRing></{'outer' if k == 0 else 'inner'}BoundaryIs>"
                  for k, anillo in enumerate(r["geometry"]["coordinates"]))
        + "</Polygon></Placemark>" for r in RASGOS)
    with open(CARPETA_CHARCOS / f"{c}.kml", "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?><kml xmlns="http://www.opengis.net/kml/2.2"><Document>'
                f'<name>charcos {c}</name><Style id="charco"><LineStyle><color>ffff00ff</color><width>1</width></LineStyle>'
                f'<PolyStyle><color>80ff00ff</color></PolyStyle></Style>{marcas}</Document></kml>')
    CHARCOS += [r["properties"] for r in RASGOS]
    paso = 10 if ciudad == "lima" else 5
    VISTA_FINAL[c] = (suave(*leer(c)[:2])[::paso, ::paso], ndimage.maximum_filter(m.astype("uint8"), paso)[::paso, ::paso],
                      permanente[::paso, ::paso], m.sum() * 25 / 1e6)
CHARCOS = pd.DataFrame(CHARCOS, columns=["escena", "charco", "lat", "lon", "area_m2"])
CHARCOS.to_csv(CARPETA_CHARCOS / "charcos.csv", index=False)
CHARCOS.groupby("escena").agg(charcos=("charco", "size"), area_km2=("area_m2", "sum")).assign(area_km2=lambda t: t.area_km2 / 1e6).round(3)

In [ ]:
fig, ejes = plt.subplots(int(np.ceil(len(VISTA_FINAL) / 4)), 4, figsize=(17, 5 * np.ceil(len(VISTA_FINAL) / 4)), squeeze=False)
for ax, (c, (fondo, charco, permanente, km2)) in zip(ejes.flat, VISTA_FINAL.items()):
    ax.imshow(fondo, cmap="gray", vmin=-25, vmax=5)
    ax.imshow(np.ma.masked_where(charco != 1, charco), cmap="cool", vmin=0, vmax=1)
    ax.contour(permanente, [0.5], colors="yellow", linewidths=0.5)
    ax.set_title(f"{c}: {km2:.3f} km²", fontsize=9)
for ax in ejes.flat:
    ax.axis("off")
plt.suptitle("charcos (rosado) y agua permanente de WorldCover (amarillo) sobre HH de ese día")
plt.tight_layout()
plt.show()

### 10.1 Valdivia: día seco vs día de lluvia

In [ ]:
SECOS_DIA = escenas.clave[escenas.dia == "seco"].tolist()
PARES = [(c, r) for c in DATOS.clave if c.startswith("valdivia_") for r in SECOS_DIA
         if r.startswith("valdivia_") and r.endswith(c.rsplit("_", 1)[1])]
permanente = WORLDCOVER["valdivia"] == 80
fuera = ~ndimage.binary_dilation(permanente, iterations=2)
AZUL = ListedColormap(["#1f6fff"])
FILAS = []
fig, ejes = plt.subplots(len(PARES), 5, figsize=(22, 4.8 * len(PARES)), squeeze=False)
for fila, (c, r) in zip(ejes, PARES):
    s, sr = suave(*leer(c)[:2]), suave(*leer(r)[:2])
    muestra = np.random.default_rng(SEMILLA).choice(s.size, 2_000_000)
    s = s - (np.nanmedian(s.flat[muestra]) - np.nanmedian(sr.flat[muestra]))
    agua_c, agua_r = agua_unet(c), agua_unet(r)
    charcos = limpiar(agua_c & fuera)
    km2 = [m.sum() * 25 / 1e6 for m in (agua_r, agua_c, charcos)]
    paneles = [(sr, None, None, f"HH día seco · {r.split('_')[1]}"), (s, None, None, f"HH día de lluvia · {c.split('_')[1]}"),
               (sr, agua_r, AZUL, f"U-Net día seco: {km2[0]:.2f} km² de agua"),
               (s, agua_c, AZUL, f"U-Net día de lluvia: {km2[1]:.2f} km² de agua"), (s, charcos, "cool", f"charcos: {km2[2]:.3f} km²")]
    for ax, (fondo, mascara, color, titulo) in zip(fila, paneles):
        ax.imshow(fondo[::3, ::3], cmap="gray", vmin=-25, vmax=5)
        if mascara is not None:
            ax.imshow(np.where(ndimage.maximum_filter(mascara.astype("uint8"), 3)[::3, ::3], 1.0, np.nan), cmap=color, vmin=0, vmax=1)
        ax.set_title(titulo, fontsize=10)
        ax.axis("off")
    fila[4].contour(permanente[::3, ::3], [0.5], colors="yellow", linewidths=0.5)
    FILAS.append({"día de lluvia": c.split("_")[1], "día seco": r.split("_")[1], "pista": c.rsplit("_", 1)[1],
                  "agua U-Net seco (km²)": km2[0], "agua U-Net lluvia (km²)": km2[1], "charcos (km²)": km2[2]})
plt.suptitle("Valdivia: día seco vs día de lluvia de la misma pista; agua de la U-Net (azul), charcos (rosado) y agua permanente (amarillo)")
plt.tight_layout(rect=(0, 0, 1, 0.975))
plt.show()
pd.DataFrame(FILAS).round(3)

## 11. Mapas de Lima Sur

Densidad de charcos en un radio de ~150 m el 5 nov y en los controles de la misma pista (día seco y garúas). El exceso es lo
que aparece solo el día de la lluvia.

In [ ]:
LIMA_SUR = (-77.02, -12.28, -76.86, -12.10)
EVENTO = escenas.clave[escenas.base == "lima_2026-11-05"].iloc[0]
MISMA_PISTA = escenas[(escenas.ciudad == "lima") & (escenas.pista == escenas.pista[escenas.clave == EVENTO].iloc[0])]
SECO = MISMA_PISTA.clave[MISMA_PISTA.dia == "seco"].iloc[0]
CONTROLES = MISMA_PISTA.clave[(MISMA_PISTA.dia == "seco") | (MISMA_PISTA.rol == "garua")].tolist()
xs, ys = Transformer.from_crs("EPSG:4326", GRILLAS["lima"]["crs"], always_xy=True).transform(LIMA_SUR[::2], LIMA_SUR[3:0:-2])
(f0, f1), (c0, c1) = rowcol(GRILLAS["lima"]["transform"], xs, ys)
f0, c0 = max(f0, 0), max(c0, 0)
SUR = np.s_[f0:f1, c0:c1]
T_SUR = GRILLAS["lima"]["transform"] * rasterio.Affine.translation(c0, f0)
PERMANENTE_SUR = ndimage.binary_dilation(WORLDCOVER["lima"] == 80, iterations=2)[SUR]
CALOR, AREA = {}, {}
for c in [EVENTO] + CONTROLES:
    with rasterio.open(f"Data/procesado/dl/predicciones/{c}.tif") as f:
        p = f.read(1)[SUR]
    evaluable = (p != 255) & ~PERMANENTE_SUR
    charco = limpiar((p >= UMBRAL * 100) & evaluable)
    peso = ndimage.gaussian_filter(evaluable.astype("float32"), 30)
    CALOR[c] = np.where(peso > 0.3, 100 * ndimage.gaussian_filter(charco.astype("float32"), 30) / np.maximum(peso, 1e-6), np.nan)
    AREA[c] = charco.sum() * 25 / 1e6
    if c == EVENTO:
        CHARCO_EVENTO = charco
HH_EVENTO, HH_SECO = suave(*leer(EVENTO)[:2])[SUR], suave(*leer(SECO)[:2])[SUR]
comun = np.isfinite(HH_EVENTO) & np.isfinite(HH_SECO) & ~PERMANENTE_SUR
CAMBIO = HH_EVENTO - np.median((HH_EVENTO - HH_SECO)[comun]) - HH_SECO
CALOR_CONTROL = np.nanmean([CALOR[c] for c in CONTROLES], axis=0)
EXCESO = CALOR[EVENTO] - CALOR_CONTROL
VMAX_CALOR = max(float(np.nanpercentile(CALOR[EVENTO], 99.5)), 0.5)
VMAX_EXCESO = max(float(np.nanpercentile(EXCESO, 99.5)), 0.5)
pd.DataFrame([{"escena": c, "rol": "evento" if c == EVENTO else ROLES[c], "charcos_km2": AREA[c],
               "densidad_media_%": np.nanmean(CALOR[c]), f"km2_densidad_>_{VMAX_CALOR / 3:.1f}_%": np.nansum(CALOR[c] > VMAX_CALOR / 3) * 25 / 1e6}
              for c in [EVENTO] + CONTROLES]).round(3)

In [ ]:
PUNTOS = NOTICIAS[NOTICIAS.escena == "lima_2026-11-05"]
fig, ejes = plt.subplots(1, 3, figsize=(19, 7.5))
paneles = [(CALOR[EVENTO], "YlOrRd", VMAX_CALOR, 0.08, f"{EVENTO}: densidad de charcos (%)"),
           (CALOR_CONTROL, "YlOrRd", VMAX_CALOR, 0.08, f"controles ({len(CONTROLES)} escenas): densidad de charcos (%)"),
           (EXCESO, "RdPu", VMAX_EXCESO, 0.15, "exceso: 5 nov − controles (puntos %)")]
for ax, (capa, cmap, vmax, corte, titulo) in zip(ejes, paneles):
    ax.imshow(HH_EVENTO[::4, ::4], cmap="gray", vmin=-25, vmax=5)
    im = ax.imshow(np.ma.masked_less(np.nan_to_num(capa[::4, ::4]), vmax * corte), cmap=cmap, vmin=0, vmax=vmax, alpha=0.85)
    plt.colorbar(im, ax=ax, shrink=0.7)
    for n in PUNTOS.itertuples():
        ax.add_patch(plt.Circle(((n.col - c0) / 4, (n.fila - f0) / 4), 25, fill=False, color="lime", lw=2))
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")
plt.suptitle("Lima Sur: mapa de calor de los charcos (radio ~150 m) y lugares de las noticias (verde)")
plt.tight_layout()
Path("Data/procesado/dl/mapas").mkdir(parents=True, exist_ok=True)
plt.savefig("Data/procesado/dl/mapas/mapa_calor_lima_sur.png", dpi=130)
plt.show()

### 11.1 Mapa interactivo

In [ ]:
alto_sur, ancho_sur = HH_EVENTO.shape
DESTINO_T, ancho_m, alto_m = calculate_default_transform(GRILLAS["lima"]["crs"], "EPSG:4326", ancho_sur, alto_sur,
                                                        *array_bounds(alto_sur, ancho_sur, T_SUR))
escala = max(ancho_m, alto_m) / 1100
if escala > 1:
    ancho_m, alto_m, DESTINO_T = int(ancho_m / escala), int(alto_m / escala), DESTINO_T * rasterio.Affine.scale(escala)
LIMITES = [[DESTINO_T.f + DESTINO_T.e * alto_m, DESTINO_T.c], [DESTINO_T.f, DESTINO_T.c + DESTINO_T.a * ancho_m]]
MAPA = folium.Map(location=[(LIMA_SUR[1] + LIMA_SUR[3]) / 2, (LIMA_SUR[0] + LIMA_SUR[2]) / 2], zoom_start=13, tiles=None,
                  control_scale=True)
folium.TileLayer("OpenStreetMap", name="Calles (OpenStreetMap)").add_to(MAPA)
folium.TileLayer("Esri.WorldImagery", name="Satélite (Esri)").add_to(MAPA)
CAPAS_MAPA = [("Radar HH · 5 nov", HH_EVENTO, "gray", -25, 5, np.isfinite(HH_EVENTO), 0.9, False),
              ("Cambio de HH respecto al día seco (dB)", CAMBIO, "RdBu", -8, 8, np.abs(CAMBIO) > 3, 0.75, False),
              ("Mapa de calor · controles", CALOR_CONTROL, "YlOrRd", 0, VMAX_CALOR, CALOR_CONTROL > 0.08 * VMAX_CALOR, 0.8, False),
              ("Mapa de calor · 5 nov", CALOR[EVENTO], "YlOrRd", 0, VMAX_CALOR, CALOR[EVENTO] > 0.08 * VMAX_CALOR, 0.8, True),
              ("Exceso el día de la lluvia", EXCESO, "RdPu", 0, VMAX_EXCESO, EXCESO > 0.15 * VMAX_EXCESO, 0.85, False),
              ("Charcos · 5 nov", CHARCO_EVENTO.astype("float32"), "spring", 0, 1, CHARCO_EVENTO, 0.95, True)]
for nombre, capa, cmap, vmin, vmax, mostrar, alfa, visible in CAPAS_MAPA:
    a = np.full((alto_m, ancho_m), np.nan, "float32")
    reproject(np.where(mostrar, capa, np.nan).astype("float32"), a, src_transform=T_SUR, src_crs=GRILLAS["lima"]["crs"],
              dst_transform=DESTINO_T, dst_crs="EPSG:4326", resampling=Resampling.average, src_nodata=np.nan, dst_nodata=np.nan)
    rgba = plt.get_cmap(cmap)(np.clip((np.nan_to_num(a) - vmin) / (vmax - vmin), 0, 1))
    rgba[..., 3] = np.where(np.isfinite(a), alfa, 0)
    folium.raster_layers.ImageOverlay((rgba * 255).astype("uint8"), bounds=LIMITES, name=nombre, show=visible).add_to(MAPA)
SITIOS = RESULTADO.set_index("lugar")
grupo = folium.FeatureGroup(name="Lugares de las noticias", show=True)
for n in PUNTOS.itertuples():
    folium.Marker([n.lat, n.lon], tooltip=n.lugar, icon=folium.Icon(color="green", icon="tint", prefix="fa"),
                  popup=folium.Popup(f"<b>{n.lugar}</b><br>{n.noticia}<br>cambio de HH: {SITIOS.loc[n.lugar, 'cambio_HH_dB']:+.2f} dB<br>"
                                     f"zonas que cambian menos: {SITIOS.loc[n.lugar, 'zonas_que_cambian_menos_%']:.0f} %", max_width=320)
                  ).add_to(grupo)
grupo.add_to(MAPA)
folium.LayerControl(collapsed=False).add_to(MAPA)
MAPA.save("Data/procesado/dl/mapas/mapa_lima_sur.html")
MAPA

## 12. Archivos de salida

In [ ]:
CARPETAS = {"modelos": sorted(Path("Data/procesado/dl").glob("unet_*.pt")) + sorted(Path("Data/procesado/ml").glob("*.joblib")),
            "tablas": sorted(Path("Data/procesado/dl").glob("*.csv")) + sorted(Path("Data/procesado").glob("*.csv")),
            "predicciones": sorted(Path("Data/procesado/dl/predicciones").glob("*.tif")),
            "mapas": sorted(Path("Data/procesado/dl/charcos").glob("*")) + sorted(Path("Data/procesado/dl/mapas").glob("*"))}
for carpeta, archivos in CARPETAS.items():
    (SALIDA / carpeta).mkdir(parents=True, exist_ok=True)
    for p in archivos:
        shutil.copy2(p, SALIDA / carpeta / p.name)
pd.DataFrame([{"carpeta": str(SALIDA / c), "archivos": len(list((SALIDA / c).iterdir())),
               "MB": sum(f.stat().st_size for f in (SALIDA / c).iterdir()) / 1e6} for c in CARPETAS]).round(1)